# API evaluation notebook

This notebook calls the MonteCarloCuda service and callback APIs and performs basic checks.

Requirements:
- The service and callback APIs are running and reachable (see `docs/USAGE.md`).
- A Kubernetes cluster is configured if you submit jobs.

Set `SERVICE_URL` and `CALLBACK_URL` in the next cell if needed.


In [ ]:
import os

try:
    import requests
except ImportError as exc:
    raise SystemExit("Install requests: pip install requests") from exc

SERVICE_URL = os.getenv("SERVICE_URL", "http://localhost:8080")
CALLBACK_URL = os.getenv("CALLBACK_URL", "http://localhost:8090")

print("SERVICE_URL:", SERVICE_URL)
print("CALLBACK_URL:", CALLBACK_URL)


In [ ]:
def check_health(base_url):
    resp = requests.get(f"{base_url}/healthz", timeout=10)
    resp.raise_for_status()
    payload = resp.json()
    assert payload.get("status") == "ok", payload
    return payload


def post_json(url, payload):
    resp = requests.post(url, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()


In [ ]:
service_health = check_health(SERVICE_URL)
callback_health = check_health(CALLBACK_URL)
service_health, callback_health


In [ ]:
job_request = {
    "simulation": "pi",
    "num_simulations": 100000,
    "mode": "single-gpu",
    "workers": 1,
    "gpus_per_worker": 1,
    "callback_url": f"{CALLBACK_URL}/callback",
}

job_response = post_json(f"{SERVICE_URL}/jobs", job_request)
expected_keys = {
    "job_name",
    "namespace",
    "mode",
    "workers",
    "gpus_per_worker",
    "nodes",
    "callback_url",
}
missing = expected_keys.difference(job_response)
assert not missing, f"Missing keys: {missing}"
job_response


In [ ]:
callback_payload = {
    "run_id": "notebook-smoke",
    "result": {"pi_estimate": 3.1415},
    "meta": {"source": "notebook"},
}

callback_response = post_json(f"{CALLBACK_URL}/callback", callback_payload)
callback_response
